**Public archival copy.** Saved cell outputs (including embedded images) and Colab-specific runtime metadata have been removed. Original code and explanatory cell content remain unchanged. Some code refers to Google Drive paths and intermediate research artifacts that are not distributed here. The Fitzpatrick17k image files and the researchers' individual access link are **not included**. See `notebooks/README.md` for access and reproducibility notes.


Phase 3 — Cross-Domain Generalization + MMD


1. Imports & Configuration

2. Load Fitzpatrick17k Metadata

3. Construct Skin-Tone Domains

4. Create Domain-Specific Train/Test Splits

5. Dataset and DataLoader

6. Load Phase 2 Baseline ResNet-50

7. Feature Extraction

8. MMD Calculation

9. Cross-Domain Experiments

10. Evaluation Metrics

11. Generalization Gap Analysis

12. Visualization

In [ ]:
# ============================================
# Phase 3 — Cross-Domain + MMD
# Imports & Configuration
# ============================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)



In [ ]:
# ============================================
# Experiment Configuration
# ============================================

SEED = 42

IMAGE_SIZE = 224
BATCH_SIZE = 32

NUM_CLASSES = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


# Reproducibility

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Load Fitzpatrick17k Metadata and Reconstruct Dataset.

The reason we rebuild instead of importing Phase 2 variables:

- Phase 3 must be independent.
- We need explicit domain separation:
- Fair domain = Fitzpatrick I–III
- Dark domain = Fitzpatrick IV–VI
- We need metadata available for cross-domain experiments.

Phase 2 used:

- Fitzpatrick17k metadata
- binary classification:
- benign = 0
- malignant = 1
- skin tone as evaluation domain only.

In [ ]:
# ============================================
# Google Drive Mount
# ============================================

from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Check MyDrive folders

import os

print(os.listdir("/content/drive/MyDrive"))

IMAGE_DIR = "/content/drive/MyDrive/finalfitz17k"

In [ ]:
# ============================================
# Count Images in Drive Folder
# ============================================

image_files = [
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(
        (".jpg",".jpeg",".png")
    )
]


print("Number of images found:", len(image_files))

print("\nFirst 5 images:")
print(image_files[:5])

In [ ]:
import pandas as pd
import numpy as np


DATASET_URL = (
    "https://raw.github.com/"
    "mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"
)

df = pd.read_csv(DATASET_URL)


model_df = df[
    (df["fitzpatrick_scale"] != -1) &
    (df["three_partition_label"].isin(
        ["benign","malignant"]
    ))
].copy()


print("Model samples:", len(model_df))

In [ ]:
# ============================================
# Verify Metadata to Image Mapping
# ============================================


available_image_hashes = {
    os.path.splitext(filename)[0]
    for filename in image_files
}


model_df["image_exists"] = (
    model_df["md5hash"]
    .isin(available_image_hashes)
)


print("Total metadata samples:",
      len(model_df))


print(
    "Images found:",
    model_df["image_exists"].sum()
)


print(
    "Images missing:",
    (~model_df["image_exists"]).sum()
)

In [ ]:
model_df["image_path"] = (
    IMAGE_DIR
    + "/"
    + model_df["md5hash"]
    + ".jpg"
)


print(model_df[
    ["md5hash","image_path"]
].head())

In [ ]:
# ============================================
# Create Skin Tone Domains
# ============================================

model_df["skin_tone"] = np.where(
    model_df["fitzpatrick_scale"].between(1,3),
    "Fair",
    "Dark"
)


# Binary target

model_df["target"] = model_df[
    "three_partition_label"
].map(
    {
        "benign":0,
        "malignant":1
    }
)


fair_df = model_df[
    model_df["skin_tone"] == "Fair"
].copy()


dark_df = model_df[
    model_df["skin_tone"] == "Dark"
].copy()


print("Fair domain:", len(fair_df))
print("Dark domain:", len(dark_df))


print("\nFair class distribution:")
print(fair_df["three_partition_label"].value_counts())


print("\nDark class distribution:")
print(dark_df["three_partition_label"].value_counts())

In [ ]:
# ============================================
# Feature Extraction Dataset
# ============================================

class FeatureDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform


    def __len__(self):
        return len(self.dataframe)


    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")


        if self.transform:
            image = self.transform(image)


        return image

Use ImageNet normalization because we use pretrained ResNet-50:

In [ ]:
# ============================================
# Feature Extraction Transform
# ============================================

feature_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [ ]:
# ============================================
# Loaders
# ============================================
fair_dataset = FeatureDataset(
    fair_df,
    transform=feature_transform
)


dark_dataset = FeatureDataset(
    dark_df,
    transform=feature_transform
)


fair_loader = DataLoader(
    fair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)


dark_loader = DataLoader(
    dark_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)


print("Fair batches:", len(fair_loader))
print("Dark batches:", len(dark_loader))

In [ ]:
# ============================================
# ResNet-50 Feature Extractor
# ============================================

weights = models.ResNet50_Weights.DEFAULT

resnet = models.resnet50(
    weights=weights
)


# Remove classification head

feature_extractor = nn.Sequential(
    *list(resnet.children())[:-1]
)


feature_extractor = feature_extractor.to(DEVICE)

feature_extractor.eval()


print(feature_extractor)

In [ ]:
# ============================================
# Feature Extraction Sanity Check
# ============================================

images = next(iter(fair_loader))

images = images.to(DEVICE)


with torch.no_grad():

    features = feature_extractor(images)


print("Raw feature shape:")
print(features.shape)

In [ ]:
features = torch.flatten(
    features,
    start_dim=1
)


print("Flattened feature shape:")
print(features.shape)

In [ ]:
# ============================================
# Extract Embeddings Function
# ============================================

def extract_embeddings(loader, model, device):

    all_features = []

    model.eval()

    with torch.no_grad():

        for images in loader:

            images = images.to(device)

            features = model(images)

            features = torch.flatten(
                features,
                start_dim=1
            )

            all_features.append(
                features.cpu().numpy()
            )


    embeddings = np.concatenate(
        all_features,
        axis=0
    )

    return embeddings

In [ ]:
# ============================================
# Fair Domain Embeddings
# ============================================

fair_embeddings = extract_embeddings(
    fair_loader,
    feature_extractor,
    DEVICE
)


print(
    "Fair embedding shape:",
    fair_embeddings.shape
)

In [ ]:
# ============================================
# Dark Domain Embeddings
# ============================================

dark_embeddings = extract_embeddings(
    dark_loader,
    feature_extractor,
    DEVICE
)


print(
    "Dark embedding shape:",
    dark_embeddings.shape
)

In [ ]:
np.save(
    "fair_embeddings.npy",
    fair_embeddings
)


np.save(
    "dark_embeddings.npy",
    dark_embeddings
)

print("Embeddings saved.")

In [ ]:
# ============================================
# Memory Efficient MMD
# ============================================

def rbf_kernel(x, y, sigma=1.0):

    xx = torch.sum(
        x ** 2,
        dim=1
    ).unsqueeze(1)

    yy = torch.sum(
        y ** 2,
        dim=1
    ).unsqueeze(0)


    distances = (
        xx + yy -
        2 * torch.mm(
            x,
            y.t()
        )
    )


    return torch.exp(
        -distances /
        (2 * sigma ** 2)
    )



def compute_mmd_rbf(
    source,
    target,
    sigma=1.0
):

    # Use CPU to avoid GPU memory issue

    source = torch.tensor(
        source,
        dtype=torch.float32
    )

    target = torch.tensor(
        target,
        dtype=torch.float32
    )


    Kxx = rbf_kernel(
        source,
        source,
        sigma
    )

    Kyy = rbf_kernel(
        target,
        target,
        sigma
    )

    Kxy = rbf_kernel(
        source,
        target,
        sigma
    )


    mmd = (
        Kxx.mean()
        +
        Kyy.mean()
        -
        2 * Kxy.mean()
    )


    return mmd.item()

In [ ]:
# ============================================
# Compute Fair-Dark MMD
# ============================================

np.random.seed(SEED)


fair_sample = fair_embeddings[
    np.random.choice(
        len(fair_embeddings),
        len(dark_embeddings),
        replace=False
    )
]


dark_sample = dark_embeddings



mmd_value = compute_mmd_rbf(
    fair_sample,
    dark_sample,
    sigma=10
)


print(
    "Fair-Dark MMD:",
    mmd_value
)

Phase 3 Result 1 — Domain Shift Quantification

The ResNet-50 feature representations of Fair and Dark skin domains are not identical. There is a measurable distribution difference in the learned feature space.

In [ ]:
# ============================================
# MMD Stability Test
# 10 Random Runs
# ============================================

mmd_scores = []


for i in range(10):

    np.random.seed(
        SEED + i
    )


    fair_sample = fair_embeddings[
        np.random.choice(
            len(fair_embeddings),
            len(dark_embeddings),
            replace=False
        )
    ]


    score = compute_mmd_rbf(
        fair_sample,
        dark_embeddings,
        sigma=10
    )


    mmd_scores.append(score)


    print(
        f"Run {i+1}: {score:.6f}"
    )


print("\nFinal MMD Result")

print(
    "Mean:",
    np.mean(mmd_scores)
)

print(
    "Std:",
    np.std(mmd_scores)
)

The result shows that the pretrained ResNet-50 representation has a measurable feature distribution difference between Fair and Dark skin domains.

The small standard deviation:

$$ \sigma = 0.00077 $$

indicates the MMD estimate is relatively stable across different random Fair-domain samples.

This is where we measure:

$$ \Delta_{gen} = Accuracy_{in-domain} - Accuracy_{cross-domain} $$

The pipeline specifically defines cross-domain testing to measure the generalization gap between domains.

For MMD we used all Fair/Dark images.

For training experiments we need:

- Train
- Validation
- Test

inside each domain.

We will maintain:

- no leakage
- same random seed
- class distribution preserved

In [ ]:
# ============================================
# Domain-Specific Train/Test Split
# ============================================

from sklearn.model_selection import train_test_split


def create_domain_split(
    dataframe,
    seed=42
):

    train_df, temp_df = train_test_split(
        dataframe,
        test_size=0.30,
        random_state=seed,
        stratify=dataframe["three_partition_label"]
    )


    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=seed,
        stratify=temp_df["three_partition_label"]
    )


    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )



fair_train_df, fair_val_df, fair_test_df = create_domain_split(
    fair_df
)


dark_train_df, dark_val_df, dark_test_df = create_domain_split(
    dark_df
)


print("Fair:")
print(
    len(fair_train_df),
    len(fair_val_df),
    len(fair_test_df)
)


print("\nDark:")
print(
    len(dark_train_df),
    len(dark_val_df),
    len(dark_test_df)
)

In [ ]:
print("Fair train")
print(
    fair_train_df["three_partition_label"]
    .value_counts()
)


print("\nDark train")
print(
    dark_train_df["three_partition_label"]
    .value_counts()
)

Domain-specific splitting is complete.

We now have a clean experimental setup.

Phase 3 Cross-Domain Split Summary
Fair Domain (Fitzpatrick I–III)
- Split	Samples
- Train	2268
- Validation	486
- Test	487


Training distribution:

- Class	Count
- Malignant	1155
- Benign	1113
- Dark Domain (Fitzpatrick IV–VI)
- Split	Samples
- Train	755
- Validation	162
- Test	162


Training distribution:

- Class	Count
- Benign	399
- Malignant	356


This is good because:

- ✅ No train/test leakage

- ✅ Stratified by disease label

- ✅ Same split logic for both domains

- ✅ Skin tone remains a domain variable only

- ✅ Model will not receive skin tone as input



Now we build the reusable training pipeline.

In [ ]:
# ============================================
# Classification Dataset
# ============================================

class ClassificationDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform


    def __len__(self):

        return len(self.dataframe)


    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]


        image = Image.open(
            row["image_path"]
        ).convert("RGB")


        if self.transform:

            image = self.transform(image)


        label = torch.tensor(
            row["target"],
            dtype=torch.long
        )


        return image, label

In [ ]:
# ============================================
# Training Transform
# ============================================

train_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [ ]:
# ============================================
# DataLoader Function
# ============================================

def create_loaders(
    train_df,
    val_df,
    test_df
):


    train_dataset = ClassificationDataset(
        train_df,
        train_transform
    )


    val_dataset = ClassificationDataset(
        val_df,
        eval_transform
    )


    test_dataset = ClassificationDataset(
        test_df,
        eval_transform
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


    return (
        train_loader,
        val_loader,
        test_loader
    )

In [ ]:
fair_loaders = create_loaders(
    fair_train_df,
    fair_val_df,
    fair_test_df
)


dark_loaders = create_loaders(
    dark_train_df,
    dark_val_df,
    dark_test_df
)

In [ ]:
# ============================================
# DataLoader Sanity Check
# ============================================

fair_train_loader, fair_val_loader, fair_test_loader = fair_loaders

dark_train_loader, dark_val_loader, dark_test_loader = dark_loaders


print("Fair domain:")
print("Train batches:", len(fair_train_loader))
print("Validation batches:", len(fair_val_loader))
print("Test batches:", len(fair_test_loader))


print("\nDark domain:")
print("Train batches:", len(dark_train_loader))
print("Validation batches:", len(dark_val_loader))
print("Test batches:", len(dark_test_loader))


# Check one batch

images, labels = next(iter(fair_train_loader))

print("\nBatch check:")
print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Labels:", labels[:10])

In [ ]:
# ============================================
# ResNet-50 Model Creator
# ============================================

def create_resnet50():

    weights = models.ResNet50_Weights.DEFAULT

    model = models.resnet50(
        weights=weights
    )


    num_features = model.fc.in_features


    model.fc = nn.Linear(
        num_features,
        NUM_CLASSES
    )


    model = model.to(DEVICE)


    return model

In [ ]:
# ============================================
# Training Function
# ============================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    running_loss = 0
    correct = 0
    total = 0


    for images, labels in loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)


        optimizer.zero_grad()


        outputs = model(images)


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()


        optimizer.step()


        running_loss += (
            loss.item()
            *
            images.size(0)
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        correct += (
            predictions == labels
        ).sum().item()


        total += labels.size(0)



    return (
        running_loss / total,
        correct / total
    )

In [ ]:
# ============================================
# Validation Function
# ============================================

def evaluate_loss(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)


            outputs = model(images)


            loss = criterion(
                outputs,
                labels
            )


            running_loss += (
                loss.item()
                *
                images.size(0)
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            correct += (
                predictions == labels
            ).sum().item()


            total += labels.size(0)


    return (
        running_loss / total,
        correct / total
    )

In [ ]:
# ============================================
# Domain Training Pipeline
# ============================================

def train_domain_model(
    train_loader,
    val_loader,
    epochs=50
):

    model = create_resnet50()


    criterion = nn.CrossEntropyLoss()


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )


    best_val_loss = np.inf
    patience = 10
    counter = 0


    best_weights = None


    for epoch in range(epochs):


        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer
        )


        val_loss, val_acc = evaluate_loss(
            model,
            val_loader,
            criterion
        )


        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )


        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_weights = (
                model.state_dict()
                .copy()
            )

            counter = 0

        else:

            counter += 1


        if counter >= patience:

            print("Early stopping")
            break



    model.load_state_dict(
        best_weights
    )


    return model

In [ ]:
# ============================================
# Evaluation Function
# ============================================

def evaluate_model(
    model,
    test_loader
):

    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []


    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(DEVICE)

            outputs = model(images)


            probabilities = torch.softmax(
                outputs,
                dim=1
            )[:,1]


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            all_labels.extend(
                labels.numpy()
            )

            all_preds.extend(
                predictions.cpu().numpy()
            )

            all_probs.extend(
                probabilities.cpu().numpy()
            )


    accuracy = accuracy_score(
        all_labels,
        all_preds
    )


    precision = precision_score(
        all_labels,
        all_preds
    )


    recall = recall_score(
        all_labels,
        all_preds
    )


    f1 = f1_score(
        all_labels,
        all_preds
    )


    auc = roc_auc_score(
        all_labels,
        all_probs
    )


    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "AUC": auc
    }

In [ ]:
# ============================================
# Experiment FF
# Fair Train → Fair Test
# ============================================

print("Training FF model...")


FF_model = train_domain_model(
    fair_train_loader,
    fair_val_loader,
    epochs=50
)


print("\nEvaluating FF...")


FF_results = evaluate_model(
    FF_model,
    fair_test_loader
)


print(FF_results)

In [ ]:
# ============================================
# Store Experiment Results
# ============================================

cross_domain_results = {}

cross_domain_results["FF"] = FF_results

cross_domain_results

In [ ]:
# ============================================
# Experiment FD
# Fair Train → Dark Test
# ============================================

print("Training FD model...")


FD_model = train_domain_model(
    fair_train_loader,
    fair_val_loader,
    epochs=50
)


print("\nEvaluating FD...")


FD_results = evaluate_model(
    FD_model,
    dark_test_loader
)


print(FD_results)


cross_domain_results["FD"] = FD_results

In [ ]:
cross_domain_results["FD"] = FD_results

Now we can calculate:

$$ \Delta_{gen}=Accuracy_{in-domain}-Accuracy_{cross-domain} $$

For Fair-trained model:

FF
> Fair → Fair
>
> Accuracy = 0.7926

FD
> Fair → Dark
>
> Accuracy = 0.7284


Therefore:

$$ 0.7926 - 0.7284 = 0.0642 $$

Fair → Dark Generalization Gap

Δgen = 6.42 percentage points

The model loses approximately 6.4% accuracy when evaluated on the Dark domain compared with the Fair domain reference.

This connects directly with the Phase 3 objective: quantify how much performance changes when the model is transferred across domains.

In [ ]:
# ============================================
# Experiment DD
# Dark Train → Dark Test
# ============================================

print("Training DD model...")


DD_model = train_domain_model(
    dark_train_loader,
    dark_val_loader,
    epochs=50
)


print("\nEvaluating DD...")


DD_results = evaluate_model(
    DD_model,
    dark_test_loader
)


print(DD_results)


cross_domain_results["DD"] = DD_results

In [ ]:
# ============================================
# Experiment DF
# Dark Train → Fair Test
# ============================================

print("Training DF model...")


DF_model = train_domain_model(
    dark_train_loader,
    dark_val_loader,
    epochs=50
)


print("\nEvaluating DF...")


DF_results = evaluate_model(
    DF_model,
    fair_test_loader
)


print(DF_results)


cross_domain_results["DF"] = DF_results

In [ ]:
cross_domain_results["DF"] = DF_results

In [ ]:
# ============================================
# Final Cross-Domain Results Table
# ============================================

results_df = pd.DataFrame(
    cross_domain_results
).T


results_df

**Generalization Gap Analysis**

Now calculate the domain transfer drops.

Fair → Dark

Reference:
> FF Accuracy = 0.7926

Transfer:
> FD Accuracy = 0.7284

$$ \Delta_{FD} = 0.7926-0.7284 $$

Result: 6.42 percentage points

Dark → Fair

Reference:
> DD Accuracy = 0.8272

Transfer:
> DF Accuracy = 0.6940

$$ \Delta_{DF} = 0.8272-0.6940 $$

Result: 13.31 percentage points

**Phase 3 Main Findings**

1.Feature distribution shift exists

MMD:
> 0.0207 ± 0.0008

This shows ResNet-50 embeddings differ between Fair and Dark domains.

2.Cross-domain performance decreases

| Direction | In-domain Accuracy | Cross-domain Accuracy | Drop |
| :--- | :---: | :---: | :---: |
| Fair → Dark | 79.26% | 72.84% | 6.42% |
| Dark → Fair | 82.72% | 69.40% | 13.31% |


3.Dark-trained model transfers less effectively to Fair images

The observed accuracy reduction is larger in Dark → Fair compared with Fair → Dark under this experimental setup.

This is a descriptive result from these experiments. Later phases will investigate whether controlled mixing and domain-invariant learning can reduce these gaps.

Phase 3 experiments are complete, so now we convert the raw outputs into a publication-ready Results & Analysis section.

This section will contain:

- Table 1 — Dataset and Domain Distribution
- Table 2 — MMD Domain Shift Quantification
- Table 3 — Cross-Domain Generalization Performance
- Table 4 — Generalization Gap Analysis
- Figure 1 — MMD Domain Shift Visualization
- Figure 2 — Cross-Domain Accuracy Comparison
- Figure 3 — Accuracy Drop Due to Domain Transfer
- Paper-style interpretation text

**Phase 3 Results and Analysis**

3.1 Domain Distribution Analysis

Before cross-domain evaluation, the dataset was divided into two skin-tone domains based on Fitzpatrick classification:

- Fair domain: Fitzpatrick I–III
- Dark domain: Fitzpatrick IV–VI

The final dataset consisted of 4,320 binary skin lesion images.

Table 1. Domain and Class Distribution
| Domain       | Total Images | Benign | Malignant |
| ------------ | -----------: | -----: | --------: |
| Fair (I–III) |         3241 |   1590 |      1651 |
| Dark (IV–VI) |         1079 |    570 |       509 |
| Total        |         4320 |   2160 |      2160 |

Interpretation

The disease classes were balanced overall, while the skin-tone domains showed substantial imbalance. Therefore, Phase 3 focused on measuring domain differences rather than applying balancing techniques.

3.2 Feature Distribution Shift Using MMD

To quantify domain shift, ImageNet-pretrained ResNet-50 was used as a feature extractor. The final classification layer was removed, producing 2048-dimensional image embeddings.

The Maximum Mean Discrepancy (MMD) between Fair and Dark feature distributions was calculated.

Table 2. MMD-Based Domain Shift Measurement
| Comparison   | Feature Extractor | Runs | Mean MMD | Standard Deviation |
| ------------ | ----------------- | ---: | -------: | -----------------: |
| Fair vs Dark | ResNet-50         |   10 |  0.02072 |            0.00077 |

Interpretation

The ResNet-50 embedding space showed a measurable discrepancy between Fair and Dark skin domains.

The low variation across 10 repeated sampling experiments indicates that the observed domain difference was stable rather than caused by random sampling.

``` Figure 1 — MMD Domain Shift Visualization ```

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(7,5))

plt.bar(
    ["Fair-Dark"],
    [np.mean(mmd_scores)]
)


plt.ylabel("MMD Value")
plt.title(
    "Feature Distribution Shift Between Skin-Tone Domains"
)

plt.text(
    0,
    np.mean(mmd_scores),
    f"{np.mean(mmd_scores):.5f}",
    ha="center",
    va="bottom"
)

plt.show()

3.3 Cross-Domain Generalization Performance

Four independent ResNet-50 models were trained from ImageNet initialization.

The experiments evaluated:

- Same-domain performance
- Cross-domain transfer performance

Table 3. Cross-Domain Classification Performance
| Experiment | Training Domain | Testing Domain | Accuracy | Precision | Recall | F1-score |    AUC |
| ---------- | --------------- | -------------- | -------: | --------: | -----: | -------: | -----: |
| FF         | Fair            | Fair           |   0.7926 |    0.7634 | 0.8589 |   0.8083 | 0.8955 |
| FD         | Fair            | Dark           |   0.7284 |    0.7500 | 0.6316 |   0.6857 | 0.7948 |
| DD         | Dark            | Dark           |   0.8272 |    0.8750 | 0.7368 |   0.8000 | 0.9129 |
| DF         | Dark            | Fair           |   0.6940 |    0.7592 | 0.5847 |   0.6606 | 0.7886 |


```Figure 2 — Cross-Domain Accuracy Comparison ```

In [ ]:
experiments = [
    "FF",
    "FD",
    "DD",
    "DF"
]


accuracy = [
    cross_domain_results[x]["Accuracy"]
    for x in experiments
]


plt.figure(figsize=(8,5))


plt.bar(
    experiments,
    accuracy
)


plt.ylabel("Accuracy")
plt.xlabel("Experiment")

plt.ylim(0,1)

plt.title(
    "Cross-Domain Skin Lesion Classification Accuracy"
)


for i,v in enumerate(accuracy):

    plt.text(
        i,
        v+0.02,
        f"{v:.3f}",
        ha="center"
    )


plt.show()

3.4 Generalization Gap Analysis

The performance reduction caused by domain transfer was calculated as:

$$ \Delta_{gen} = Accuracy_{in-domain} - Accuracy_{cross-domain} $$

Table 4. Domain Transfer Performance Drop
| Transfer Direction | In-Domain Accuracy | Cross-Domain Accuracy | Generalization Gap |
| ------------------ | -----------------: | --------------------: | -----------------: |
| Fair → Dark        |             0.7926 |                0.7284 |             0.0642 |
| Dark → Fair        |             0.8272 |                0.6940 |             0.1331 |

```Figure 3 — Generalization Gap ```

In [ ]:
import matplotlib.pyplot as plt

directions = [
    "Fair → Dark",
    "Dark → Fair"
]

gap = [
    0.0642,
    0.1331
]

plt.figure(figsize=(7,5))

plt.bar(
    directions,
    gap
)

plt.ylabel(
    "Accuracy Drop"
)

plt.title(
    "Cross-Domain Generalization Gap"
)

# FIX: Set the Y-axis limit higher so text doesn't collide with the top line
plt.ylim(0, 0.16)

for i, v in enumerate(gap):
    plt.text(
        i,
        v + 0.005,
        f"{v:.3f}",
        ha="center"
    )

plt.show()


**Phase 3 Discussion (Paper Style)**

Domain Shift

- The MMD analysis demonstrated that ResNet-50 feature representations differ between Fair and Dark skin-tone domains. The consistent MMD value across repeated trials suggests that the domain discrepancy exists within the learned feature space.

Cross-Domain Transfer

- The in-domain experiments (FF and DD) achieved higher performance compared with cross-domain experiments (FD and DF). When models were evaluated on an unseen skin-tone domain, classification accuracy decreased.

- The Fair-trained model experienced a reduction from 79.26% to 72.84% when transferred to Dark images, representing a 6.42 percentage-point decrease. Similarly, the Dark-trained model decreased from 82.72% to 69.40% when evaluated on Fair images, producing a 13.31 percentage-point decrease.

These findings demonstrate that skin-tone domain variation affects model generalization and motivates the need for domain-invariant learning approaches in later phases.